In [1]:
import pandas as pd
import os
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

scale_to_int = {
    "10k": 10000, "100k": 100000, "1million": 1000000,
    "10million": 10000000, "100million": 100000000, "250million": 250000000
}

# This function takes a list of files (1 or many), averages them, and returns the stats
def process_test_files(file_list, folder_path, test_type, num_elements):
    np_runs, ocl_comp_runs, ocl_trans_runs = [], [], []
    
    for file in file_list:
        try:
            df = pd.read_json(os.path.join(folder_path, file))
            np_runs.append(df['numpy_time'])
            ocl_comp_runs.append(df['opencl_time'])
            ocl_trans_runs.append(df['opencl_transfer_time'])
        except Exception as e:
            print(f"Error reading {file}: {e}")
            return None
            
    if not np_runs: return None
    
    # Average the runs (handles both 3 files for Basic and 1 file for Poly perfectly)
    np_avg = (sum(np_runs) / len(np_runs)).mean()
    ocl_comp_avg = (sum(ocl_comp_runs) / len(ocl_comp_runs)).mean()
    ocl_trans_avg = (sum(ocl_trans_runs) / len(ocl_trans_runs)).mean()
    
    ocl_total_avg = ocl_comp_avg + ocl_trans_avg
    
    return {
        'operation': test_type,
        'number_of_elements': num_elements,
        'numpy_avg': np_avg, 
        'opencl_compute_avg': ocl_comp_avg,
        'opencl_transfer_avg': ocl_trans_avg,
        'opencl_total_avg': ocl_total_avg,
        'speedup_total_x': np_avg / ocl_total_avg,
        'speedup_compute_x': np_avg / ocl_comp_avg
    }

In [2]:
file_blocks = [
  ["add1st10ktest.json", "add2nd10ktest.json", "add3rd10ktest.json"],
  ["add1st100ktest.json", "add2nd100ktest.json", "add3rd100ktest.json"],
  ["add1st1millionTest.json", "add2nd1millionTest.json", "add3rd1millionTest.json"],
  ["add1st10millionTest.json", "add2nd10millionTest.json", "add3rd10millionTest.json"],
  ["add1st100millionTest.json", "add2nd100millionTest.json", "add3rd100millionTest.json"],
  ["mul1st10ktest.json", "mul2nd10ktest.json", "mul3rd10ktest.json"],
  ["mul1st100ktest.json", "mul2nd100ktest.json", "mul3rd100ktest.json"],
  ["mul1st1millionTest.json", "mul2nd1millionTest.json", "mul3rd1millionTest.json"],
  ["mul1st10millionTest.json", "mul2nd10millionTest.json", "mul3rd10millionTest.json"],
  ["mul100millionTest.json", "mul2nd100millionTest.json", "mul3rd100millionTest.json"],
  ["mul1st250millionTest.json", "mul2nd250millionTest.json", "mul3rd250millionTest.json"],
  ["sub1st10ktest.json", "sub2nd10ktest.json", "sub3rd10ktest.json"],
  ["sub1st100ktest.json", "sub2nd100ktest.json", "sub3rd100ktest.json"],
  ["sub1st1millionTest.json", "sub2nd1millionTest.json", "sub3rd1millionTest.json"],
  ["sub1st10millionTest.json", "sub2nd10millionTest.json", "sub3rd10millionTest.json"],
  ["sub1st100millionTest.json", "sub2nd100millionTest.json", "sub3rd100millionTest.json"]
]

basic_summary = []

for block in file_blocks:
    # Quick string parsing to get the operation and scale
    base_name = block[0].replace("1st", "").replace("Test.json", "").replace("test.json", "")
    op = "add" if "add" in base_name else "mul" if "mul" in base_name else "sub"
    scale_str = base_name.replace(op, "")
    
    # Call our function
    result = process_test_files(block, "testDataForAverageTime", op, scale_to_int[scale_str])
    if result: basic_summary.append(result)

results_df = pd.DataFrame(basic_summary)

df_add = results_df[results_df['operation'] == 'add'].drop(columns=['operation']).sort_values('number_of_elements').reset_index(drop=True)
df_sub = results_df[results_df['operation'] == 'sub'].drop(columns=['operation']).sort_values('number_of_elements').reset_index(drop=True)
df_mul = results_df[results_df['operation'] == 'mul'].drop(columns=['operation']).sort_values('number_of_elements').reset_index(drop=True)

print("Basic Operations processed and separated!")

Basic Operations processed and separated!


In [3]:
poly_files = [
    "poly10kelements.json", "poly100kelements.json", "poly1millionElements.json",
    "poly10millionElements.json", "poly100millionElements.json", "poly250millionElements.json"
]

poly_summary = []

for file in poly_files:
    scale_str = file.replace("poly", "").replace("Elements.json", "").replace("elements.json", "")
    
    # Pass it as a list of 1 file. The function handles it perfectly.
    result = process_test_files([file], "testDataForPoly", "poly", scale_to_int[scale_str])
    if result: poly_summary.append(result)

df_poly = pd.DataFrame(poly_summary).drop(columns=['operation']).sort_values('number_of_elements').reset_index(drop=True)

print("Polynomials processed!")

Polynomials processed!


In [4]:
print("--- ADDITION ---"); display(df_add)
print("\n--- SUBTRACTION ---"); display(df_sub)
print("\n--- MULTIPLICATION ---"); display(df_mul)
print("\n--- POLYNOMIAL ---"); display(df_poly)

--- ADDITION ---


,number_of_elements,numpy_avg,opencl_compute_avg,opencl_transfer_avg,opencl_total_avg,speedup_total_x,speedup_compute_x
0,10000,0.000009,0.000003,0.004359,0.004362,0.001988,2.823750
1,100000,0.000054,0.000009,0.005860,0.005869,0.009255,6.019156
2,1000000,0.001952,0.000033,0.005096,0.005129,0.380583,58.456091
3,10000000,0.019043,0.001167,0.047025,0.048192,0.395156,16.314471
4,100000000,0.215411,0.012007,0.526113,0.538120,0.400303,17.940615



--- SUBTRACTION ---


,number_of_elements,numpy_avg,opencl_compute_avg,opencl_transfer_avg,opencl_total_avg,speedup_total_x,speedup_compute_x
0,10000,0.000008,0.000005,0.000512,0.000517,0.014544,1.405074
1,100000,0.000062,0.000008,0.001113,0.001121,0.055329,7.531212
2,1000000,0.001919,0.000033,0.005689,0.005722,0.335459,58.180088
3,10000000,0.019190,0.001167,0.049359,0.050527,0.379804,16.439732
4,100000000,0.193770,0.011993,0.509500,0.521493,0.371569,16.157412



--- MULTIPLICATION ---


,number_of_elements,numpy_avg,opencl_compute_avg,opencl_transfer_avg,opencl_total_avg,speedup_total_x,speedup_compute_x
0,10000,0.000008,0.000005,0.000631,0.000636,0.012553,1.687109
1,100000,0.000047,0.000009,0.001149,0.001158,0.040360,4.938855
2,1000000,0.001990,0.000033,0.005387,0.005420,0.367088,59.759339
3,10000000,0.020782,0.001168,0.050344,0.051512,0.403442,17.795670
4,100000000,0.210796,0.011994,0.513963,0.525958,0.400785,17.574635
5,250000000,0.507097,0.030276,1.238435,1.268711,0.399694,16.749027



--- POLYNOMIAL ---


,number_of_elements,numpy_avg,opencl_compute_avg,opencl_transfer_avg,opencl_total_avg,speedup_total_x,speedup_compute_x
0,10000,0.000094,0.000002,0.000662,0.000664,0.142264,53.677273
1,100000,0.001160,0.000006,0.003813,0.003819,0.303605,179.928159
2,1000000,0.014185,0.000042,0.003376,0.003418,4.149932,336.050128
3,10000000,0.146450,0.001208,0.033641,0.034849,4.202378,121.209070
4,100000000,1.518668,0.012739,0.584225,0.596964,2.543986,119.215723
5,250000000,3.758398,0.031464,1.291369,1.322833,2.841174,119.450851


In [5]:
# Save each DataFrame to a CSV file. 
# index=False prevents Pandas from writing the row numbers (0, 1, 2...) into the file
df_add.to_csv("addition_results.csv", index=False)
df_sub.to_csv("subtraction_results.csv", index=False)
df_mul.to_csv("multiplication_results.csv", index=False)
df_poly.to_csv("polynomial_results.csv", index=False)

print("✅ All 4 DataFrames successfully saved to CSV files!")

✅ All 4 DataFrames successfully saved to CSV files!
